# doSnowflake -- Parallel Processing with foreach

This notebook demonstrates the **doSnowflake** parallel backend for R's
`foreach` package. It enables `%dopar%` loops to execute on Snowflake compute.

**Phase 0** provides local (in-container) parallelism using socket clusters,
giving you multi-core execution inside a Workspace Notebook or SPCS container.

**Sections:**
1. Setup
2. Basic `foreach %dopar%` usage
3. Hyperparameter tuning example
4. Sequential vs parallel timing comparison

## 1. Setup

Install the R environment and register the `%%R` magic. If you have already
run this in another notebook during this session, you can skip this cell.

In [ ]:
from sfnb_setup import install_r
install_r()

In [ ]:
from sfnb_setup import install_r_packages
install_r_packages(packages=["snowflakeR"])

In [ ]:
%%R
library(snowflakeR)
library(foreach)

conn <- sfr_connect()

## 2. Basic foreach %dopar% Usage

Register the doSnowflake backend and run a simple parallel loop.
`registerDoSnowflake()` auto-detects available CPU cores and reserves
one for the main R thread.

In [ ]:
%%R
registerDoSnowflake(conn)

cat("Backend:", foreach::getDoParName(), "\n")
cat("Workers:", foreach::getDoParWorkers(), "\n")

In [ ]:
%%R
# Simple parallel computation: square each number
result <- foreach(i = 1:20, .combine = c) %dopar% {
  Sys.sleep(0.1)  # simulate work
  i^2
}

cat("Results:", result, "\n")

In [ ]:
%%R
# Combine results into a data.frame
grid_results <- foreach(
  alpha = c(0.01, 0.1, 0.5, 1.0),
  .combine = rbind
) %dopar% {
  data.frame(alpha = alpha, score = 1 / (1 + alpha))
}

print(grid_results)

## 3. Hyperparameter Tuning Example

A practical use case: grid search over `randomForest` hyperparameters
using the `iris` dataset. Each parameter combination trains a model
and evaluates out-of-sample accuracy on a parallel worker.

In [ ]:
%%R
if (!requireNamespace("randomForest", quietly = TRUE)) {
  install.packages("randomForest", repos = "https://cloud.r-project.org")
}

# Define a hyperparameter grid
param_grid <- expand.grid(
  ntree  = c(50, 100, 200, 500),
  mtry   = c(1, 2, 3, 4),
  stringsAsFactors = FALSE
)

cat("Grid size:", nrow(param_grid), "parameter combinations\n")
print(param_grid)

In [ ]:
%%R
# Train-test split
set.seed(42)
train_idx <- sample(nrow(iris), 0.7 * nrow(iris))
train_data <- iris[train_idx, ]
test_data  <- iris[-train_idx, ]

# Parallel grid search
registerDoSnowflake(conn)

t_start <- proc.time()

hpo_results <- foreach(
  idx = seq_len(nrow(param_grid)),
  .combine  = rbind,
  .packages = "randomForest",
  .export   = c("param_grid", "train_data", "test_data")
) %dopar% {
  params <- param_grid[idx, ]

  model <- randomForest::randomForest(
    Species ~ ., data = train_data,
    ntree = params$ntree,
    mtry  = params$mtry
  )

  preds    <- predict(model, test_data)
  accuracy <- mean(preds == test_data$Species)

  data.frame(
    ntree    = params$ntree,
    mtry     = params$mtry,
    accuracy = accuracy
  )
}

t_parallel <- (proc.time() - t_start)["elapsed"]

# Show results sorted by accuracy
hpo_results <- hpo_results[order(-hpo_results$accuracy), ]
cat("\nHPO Results (sorted by accuracy):\n")
print(hpo_results)
cat("\nBest configuration:\n")
print(hpo_results[1, ])
cat(sprintf("\nParallel time: %.2f seconds\n", t_parallel))

## 4. Sequential vs Parallel Timing Comparison

Compare wall-clock time between sequential and parallel execution
to see the speedup from multi-core processing.

In [ ]:
%%R
n_tasks <- 20
work_time <- 0.2  # seconds of simulated work per task

# --- Sequential ---
registerDoSnowflake(conn, workers = 1L)
t1 <- proc.time()
seq_result <- foreach(i = seq_len(n_tasks), .combine = c) %dopar% {
  Sys.sleep(work_time)
  i
}
t_seq <- (proc.time() - t1)["elapsed"]

# --- Parallel (auto workers) ---
registerDoSnowflake(conn)
n_workers <- foreach::getDoParWorkers()
t2 <- proc.time()
par_result <- foreach(i = seq_len(n_tasks), .combine = c) %dopar% {
  Sys.sleep(work_time)
  i
}
t_par <- (proc.time() - t2)["elapsed"]

# --- Report ---
cat(sprintf("Tasks: %d | Work per task: %.1fs\n", n_tasks, work_time))
cat(sprintf("Sequential:  %.2fs (1 worker)\n", t_seq))
cat(sprintf("Parallel:    %.2fs (%d workers)\n", t_par, n_workers))
cat(sprintf("Speedup:     %.1fx\n", t_seq / t_par))

# Verify correctness
stopifnot(identical(seq_result, par_result))
cat("\nResults match: TRUE\n")